In [1]:
!pip install -q pypdf sentence-transformers faiss-cpu groq

In [2]:
from google.colab import files

uploaded = files.upload()

Saving RAG_Sample_Test_Document.pdf to RAG_Sample_Test_Document.pdf


In [3]:
from pypdf import PdfReader

pdf_path = list(uploaded.keys())[0]

reader = PdfReader(pdf_path)

pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text()

    if text and text.strip():
        pages.append({
            "page": page_number,
            "text": text
        })

print("PDF extracted successfully!")
print("Total pages:", len(pages))
print("\nSample text:")
print(pages[0]["text"][:1000])

PDF extracted successfully!
Total pages: 2

Sample text:
RAG Sample Test Document
A short document for testing PDF upload, text extraction, retrieval, and question answering in a
Retrieval-Augmented Generation (RAG) notebook.
1. What is Retrieval-Augmented Generation?
Retrieval-Augmented Generation, commonly called RAG, is an approach that combines information
retrieval with text generation. Before answering a question, the system searches an external
knowledge source for relevant information. The retrieved information is then provided to a language
model as context.
2. Main Components
Document collection: The source files that contain useful information, such as PDF documents or
text files.
Chunking: Splitting long documents into smaller passages. Chunking helps the system retrieve
focused information instead of processing an entire document at once.
Embedding model: A model that converts text into numerical vectors. Texts with similar meanings
are expected to have similar vector repr

##Split the PDF into smaller chunks

In [4]:
def split_pages_into_chunks(pages, chunk_size=500, overlap=100):
    chunks = []
    step = chunk_size - overlap

    for page in pages:
        words = page["text"].split()

        for i in range(0, len(words), step):
            chunk_text = " ".join(words[i:i + chunk_size])

            if chunk_text.strip():
                chunks.append({
                    "text": chunk_text,
                    "page": page["page"]
                })

    return chunks

chunks = split_pages_into_chunks(pages)

print("Total chunks:", len(chunks))
print("\nFirst chunk:")
print(chunks[0]["text"])

Total chunks: 2

First chunk:
RAG Sample Test Document A short document for testing PDF upload, text extraction, retrieval, and question answering in a Retrieval-Augmented Generation (RAG) notebook. 1. What is Retrieval-Augmented Generation? Retrieval-Augmented Generation, commonly called RAG, is an approach that combines information retrieval with text generation. Before answering a question, the system searches an external knowledge source for relevant information. The retrieved information is then provided to a language model as context. 2. Main Components Document collection: The source files that contain useful information, such as PDF documents or text files. Chunking: Splitting long documents into smaller passages. Chunking helps the system retrieve focused information instead of processing an entire document at once. Embedding model: A model that converts text into numerical vectors. Texts with similar meanings are expected to have similar vector representations. Vector index: 

##Generate Text Embeddings


In [5]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

print("Embeddings generated successfully!")
print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generated successfully!
Embedding shape: (2, 384)


##Create Faiss vector index

In [6]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS index created successfully!")
print("Total vectors stored:", index.ntotal)

FAISS index created successfully!
Total vectors stored: 2


In [7]:
def retrieve_documents(question, top_k=3):
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        question_embedding,
        min(top_k, len(chunks))
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            results.append({
                "text": chunks[idx]["text"],
                "page": chunks[idx]["page"],
                "score": float(score)
            })

    return results


question = "What is the role of FAISS in a RAG system?"

retrieved_docs = retrieve_documents(question)

for doc in retrieved_docs:
    print("Page:", doc["page"])
    print("Similarity:", round(doc["score"], 3))
    print(doc["text"][:500])
    print("-" * 50)

Page: 2
Similarity: 0.54
1. What does RAG stand for? 2. Why are documents divided into chunks? 3. What is the role of an embedding model? 4. How does FAISS help a RAG system? 5. Name two limitations of RAG.
--------------------------------------------------
Page: 1
Similarity: 0.348
RAG Sample Test Document A short document for testing PDF upload, text extraction, retrieval, and question answering in a Retrieval-Augmented Generation (RAG) notebook. 1. What is Retrieval-Augmented Generation? Retrieval-Augmented Generation, commonly called RAG, is an approach that combines information retrieval with text generation. Before answering a question, the system searches an external knowledge source for relevant information. The retrieved information is then provided to a language m
--------------------------------------------------


In [8]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

print("Groq connected successfully!")

Groq connected successfully!


In [9]:
def generate_answer(question, retrieved_docs):

    context = "\n\n".join(
        f"Page {doc['page']}:\n{doc['text']}"
        for doc in retrieved_docs
    )

    prompt = f"""
    Answer the question using only the provided context.

    If the answer is not available in the context,
    clearly say that the information is not available.

    Context:
    {context}

    Question:
    {question}
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful document question-answering assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_completion_tokens=1024
    )

    return response.choices[0].message.content

In [10]:
question = "What is the role of FAISS in a RAG system?"

retrieved_docs = retrieve_documents(question)

answer = generate_answer(question, retrieved_docs)

print("QUESTION:")
print(question)

print("\nGENERATED ANSWER:")
print(answer)

print("\nSOURCE REFERENCES:")
for doc in retrieved_docs:
    print(f"Page {doc['page']} | Similarity: {doc['score']:.3f}")

QUESTION:
What is the role of FAISS in a RAG system?

GENERATED ANSWER:
FAISS is a library used for efficient vector search. In a RAG system it stores the embeddings of document chunks in a vector index and enables fast similarity search to retrieve the most relevant chunks for a user’s question.

SOURCE REFERENCES:
Page 2 | Similarity: 0.540
Page 1 | Similarity: 0.348


In [11]:
test_questions = [
    "What does RAG stand for?",
    "Why are documents divided into chunks?",
    "What is the role of an embedding model?",
    "How does FAISS help a RAG system?",
    "What are the limitations of RAG?"
]

evaluation_results = []

for question in test_questions:

    retrieved_docs = retrieve_documents(question)
    answer = generate_answer(question, retrieved_docs)

    evaluation_results.append({
        "Question": question,
        "Answer": answer,
        "Retrieved Pages": sorted(set(
            doc["page"] for doc in retrieved_docs
        ))
    })

    print("\nQUESTION:", question)
    print("ANSWER:", answer)
    print("SOURCE PAGES:", sorted(set(
        doc["page"] for doc in retrieved_docs
    )))
    print("-" * 70)


QUESTION: What does RAG stand for?
ANSWER: RAG stands for **Retrieval‑Augmented Generation**.
SOURCE PAGES: [1, 2]
----------------------------------------------------------------------

QUESTION: Why are documents divided into chunks?
ANSWER: Documents are divided into chunks so that the system can retrieve focused, relevant information rather than having to process an entire long document at once. This improves the efficiency and precision of the retrieval step.
SOURCE PAGES: [1, 2]
----------------------------------------------------------------------

QUESTION: What is the role of an embedding model?
ANSWER: The embedding model converts text into numerical vectors, enabling the system to compare and retrieve passages that are semantically similar to a user’s query.
SOURCE PAGES: [1, 2]
----------------------------------------------------------------------

QUESTION: How does FAISS help a RAG system?
ANSWER: FAISS is a library used for efficient vector search. In a RAG system it po

In [12]:
import pandas as pd

results_df = pd.DataFrame(evaluation_results)

display(results_df[["Question", "Retrieved Pages"]])

,Question,Retrieved Pages
0,What does RAG stand for?,"[1, 2]"
1,Why are documents divided into chunks?,"[1, 2]"
2,What is the role of an embedding model?,"[1, 2]"
3,How does FAISS help a RAG system?,"[1, 2]"
4,What are the limitations of RAG?,"[1, 2]"


## RAG System Architecture

The implemented system follows a Retrieval-Augmented Generation architecture.

**Workflow:**

PDF Document → Text Extraction → Text Chunking → Embedding Generation → FAISS Vector Database → Similarity Search → Retrieved Context → Groq LLM → Generated Answer

**Main Components:**
1. PDF Extractor – extracts text from the uploaded PDF.
2. Text Chunker – divides extracted text into smaller sections.
3. Embedding Model – converts text into numerical vectors using Sentence Transformers.
4. FAISS Index – stores vectors and performs similarity search.
5. Retriever – selects the most relevant text chunks for a question.
6. Generator – uses Groq AI to generate an answer from retrieved context.

## Methodology

1. **Document Upload:** A PDF document is uploaded as the knowledge source.
2. **Text Extraction:** The PyPDF library extracts text from each page.
3. **Chunking:** The extracted text is divided into chunks of 500 words with 100 words of overlap.
4. **Embedding Generation:** The all-MiniLM-L6-v2 model converts each chunk into a vector representation.
5. **Vector Indexing:** FAISS stores the embeddings using an inner-product similarity index.
6. **Retrieval:** The user's question is converted into an embedding, and the top three relevant chunks are retrieved.
7. **Answer Generation:** Groq's language model generates an answer using the retrieved document context.

## Results and Observations

The implemented RAG system successfully demonstrates document-based question answering.

- The system extracts text from PDF documents.
- Text is divided into smaller overlapping chunks.
- Sentence Transformers generates embeddings for the chunks.
- FAISS retrieves relevant information based on semantic similarity.
- Groq generates answers using the retrieved context.
- Source page references help identify where information was retrieved from.

The evaluation questions and retrieval-time measurements are recorded in the generated evaluation results file. The quality of answers depends on the document content and retrieval relevance.

## Conclusion

This project implements a basic Retrieval-Augmented Generation system for PDF-based question answering. It combines document processing, text embeddings, FAISS similarity search, and Groq-based answer generation.

The implementation demonstrates how external document information can be retrieved and supplied to a language model to generate context-based answers. It provides a practical understanding of the working methodology of RAG systems.